# 10 — Unified evaluation

Load the saved final predictions, check their IDs and labels against the fixed 178,083-row test set, and recalculate the main metrics and confusion matrices. No model is trained here.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "data/raw/train.csv").exists()
)
RESULTS = ROOT / "results"
REPRODUCED = ROOT / "reproduced_runs"

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

TEST = ROOT / "data/splits/full/test.csv"
EXPECTED_TEST_ROWS = 178083
MODEL_NAMES = ["logistic_regression", "linear_svm", "distilbert", "bert_base", "hatebert"]

def load_predictions(path):
    frame = pd.read_csv(path)
    true_column = "label" if "label" in frame.columns else "true_label"
    pred_column = next(
        column for column in ["predicted_label", "prediction", "pred_label"]
        if column in frame.columns
    )
    frame = frame.rename(columns={true_column: "true_label", pred_column: "predicted_label"})
    required = {"id", "true_label", "predicted_label"}
    assert required.issubset(frame.columns)
    assert len(frame) == EXPECTED_TEST_ROWS
    assert frame["id"].is_unique
    assert set(frame["true_label"].unique()).issubset({0, 1})
    assert set(frame["predicted_label"].unique()).issubset({0, 1})
    return frame

def evaluate(frame):
    y_true = frame["true_label"].to_numpy(dtype="int8")
    y_pred = frame["predicted_label"].to_numpy(dtype="int8")
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    return {
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "toxic_f1": f1_score(y_true, y_pred, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

def evaluate_experiment(experiment):
    files = {
        model: REPRODUCED / experiment / model / "predictions.csv"
        for model in MODEL_NAMES
    }
    available = {model: path for model, path in files.items() if path.exists()}
    if not available:
        print(f"No {experiment} reproduced predictions found. Set RUN_EVALUATION=True after predictions are available.")
        return pd.DataFrame()
    canonical = pd.read_csv(TEST, usecols=["id", "label"])
    rows = []
    for model, path in available.items():
        frame = load_predictions(path).set_index("id").reindex(canonical["id"])
        assert frame["predicted_label"].notna().all()
        assert frame["true_label"].astype("int8").equals(canonical.set_index("id")["label"].astype("int8"))
        frame = frame.reset_index()
        metrics = evaluate(frame)
        rows.append({"model": model, **metrics})
        if WRITE_REPRODUCED_OUTPUTS:
            output_dir = REPRODUCED / "unified_evaluation" / experiment / model
            output_dir.mkdir(parents=True, exist_ok=True)
            (output_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
            pd.DataFrame([[metrics["tn"], metrics["fp"]], [metrics["fn"], metrics["tp"]]],
                         index=["true_0", "true_1"], columns=["pred_0", "pred_1"]).to_csv(
                             output_dir / "confusion_matrix.csv"
                         )
            frame[(frame["true_label"] == 0) & (frame["predicted_label"] == 1)].to_csv(
                output_dir / "false_positives.csv", index=False
            )
            frame[(frame["true_label"] == 1) & (frame["predicted_label"] == 0)].to_csv(
                output_dir / "false_negatives.csv", index=False
            )
    comparison = pd.DataFrame(rows)
    if WRITE_REPRODUCED_OUTPUTS and not comparison.empty:
        comparison.to_csv(REPRODUCED / "unified_evaluation" / experiment / "comparison.csv", index=False)
    return comparison

def evaluate_combined_predictions(path):
    frame = pd.read_csv(path)
    canonical = pd.read_csv(TEST, usecols=["id", "label"]).set_index("id")
    assert len(frame) == EXPECTED_TEST_ROWS and frame["id"].is_unique
    aligned = frame.set_index("id").reindex(canonical.index)
    assert np.array_equal(
        aligned["true_label"].astype("int8").to_numpy(),
        canonical["label"].astype("int8").to_numpy(),
    )
    names = {
        "LR_pred": "Logistic Regression",
        "SVM_pred": "Linear SVM",
        "DistilBERT_pred": "DistilBERT",
        "BERT_pred": "BERT-base",
        "HateBERT_pred": "HateBERT",
    }
    rows = []
    for prediction_column, model_name in names.items():
        one_model = aligned[["true_label", prediction_column]].rename(
            columns={prediction_column: "predicted_label"}
        )
        rows.append({"model": model_name, **evaluate(one_model.reset_index())})
    return pd.DataFrame(rows)

historical_predictions = {
    "fixed_200k": RESULTS / "error_analysis_full/all_model_predictions_fixed_200k.csv",
    "full_data": RESULTS / "error_analysis_full/all_model_predictions_full.csv",
}
for scope, path in historical_predictions.items():
    print(scope, "recomputed from final predictions")
    display(evaluate_combined_predictions(path))

RUN_EVALUATION = False
WRITE_REPRODUCED_OUTPUTS = False
if RUN_EVALUATION:
    for experiment in ["200k", "full"]:
        display(evaluate_experiment(experiment))

fixed_200k recomputed from final predictions


,model,tn,fp,fn,tp,accuracy,precision,recall,toxic_f1,macro_f1
0,Logistic Regression,150832,13006,3735,10510,0.905993,0.446930,0.737803,0.556659,0.752041
1,Linear SVM,152765,11073,4851,9394,0.910581,0.458983,0.659459,0.541254,0.745858
2,DistilBERT,160286,3552,5358,8887,0.949967,0.714446,0.623868,0.666092,0.819525
3,BERT-base,159979,3859,5023,9222,0.950124,0.704992,0.647385,0.674962,0.823976
4,HateBERT,159266,4572,4614,9631,0.948417,0.678096,0.676097,0.677095,0.824532


full_data recomputed from final predictions


,model,tn,fp,fn,tp,accuracy,precision,recall,toxic_f1,macro_f1
0,Logistic Regression,148727,15111,2692,11553,0.900030,0.433281,0.811021,0.564815,0.754172
1,Linear SVM,148085,15753,2951,11294,0.894970,0.417569,0.792840,0.547031,0.743815
2,DistilBERT,160596,3242,5018,9227,0.953617,0.739995,0.647736,0.690799,0.832863
3,BERT-base,161002,2836,5231,9014,0.954701,0.760675,0.632783,0.690860,0.833210
4,HateBERT,160664,3174,5020,9225,0.953988,0.744012,0.647596,0.692464,0.833799
